# Mission Logs: Rebuilding Mary's Ears
## Epoch Zero Bootcamp · NLP · Section 1: NLTK and Text Processing · Assignment 1


### Captain's briefing
You are still aboard the Hail Mary. Mary, the ship's Central AI, can fly the ship, but her
language module is damaged. She hears the galaxy, yet every message arrives as noise:
chopped words, strange symbols, broken tags.

To bring her back, you will complete two Mission Logs:

Mission Log 1:  Boot Sequence

Mission Log 2:  Decoding the Galaxy

Everything you need is on the Section 1 Notion Page. No other resources are required.

#### Mary says: "Crew member, every transmission I give you is unique to you. Nobody else's log looks like yours. Do not try to cheat. I will know."


Rules of engagement
1. Get your mission data. Download the file named mission_data.json from the Data
Vault link on the bootcamp page.
2. Work in Google Colab. Share your notebook with view access to anyone with the link
and keep its version history intact (do not copy the final code into a fresh notebook).
3. Submit three things:
    1. Your Colab notebook link (all outputs visible) and the downloaded .ipynb file (File > Download > .ipynb).
    2. Your answers_basic and/or answers_advanced dictionaries (printed at the end of your notebook, pasted into the submission form).
    3. The Captain's Debrief (below), inside the notebook as markdown cells.
5. Time. You have very limited time. The logs are short; starting early is the best strategy.
6. Integrity. This is a checkpoint of your own understanding, not a test of how well you can prompt an AI tool.
    
    Submissions will be reviewed for authenticity.
    
    Using an AI tool or a friend to understand a concept is fine, but you must declare any help in the Debrief. Having someone or something write your solution is not.
    
    Honest declarations are never penalised. Undeclared help that is discovered is.

In [8]:
# Boot-up cell (run this first)

import nltk

import json, re, math, nltk

for r in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'vader_lexicon']:
    nltk.download(r, quiet=True)

with open('mission_data.json') as f:
    data = json.load(f)
    
# upload your file to Colab first
print('NLTK version:', nltk.__version__)
print(list(data['basic'].keys()))
print(list(data['advanced'].keys()))

NLTK version: 3.9.1
['paragraph', 'words', 'docs', 'target', 'roles', 'vectors']
['log', 'train', 'test']


The output of this cell (NLTK version) must stay visible in your notebook.

Build one Python dictionary per log, using exactly the key names given in each task. Print
it at the end with print(json.dumps(answers_basic)) . Tuples become lists in JSON;
that is fine.

Rounding: round decimals to 3 places.

# MISSION LOG 1: Boot Sequence (Basic)

### Mary says: "My ears are broken. Before I can understand anything, teach me to split, clean and count words. Let's start small."

#### Your personal data lives in data['basic'] .

# B1. Ear check (tokenization)

    para = data["basic"]["paragraph"]

Report:
    
    b1_sentences : number of sentences found by sent_tokenize
    
    b1_split_tokens : number of tokens from para.split()

    b1_nltk_tokens : number of tokens from word_tokenize

In [9]:
from nltk.tokenize import sent_tokenize, word_tokenize

para = data["basic"]["paragraph"]

b1_sentences = sent_tokenize(para)
b1_split_tokens = para.split()
b1_nltk_tokens = word_tokenize(para)

print("Sentences:", len(b1_sentences))
print("Whitespace tokens:", len(b1_split_tokens))
print("NLTK tokens:", len(b1_nltk_tokens))

Sentences: 3
Whitespace tokens: 33
NLTK tokens: 41


# B2. Noise filter (cleaning)

Apply exactly this recipe to para , in this order:

    1. lowercase

    2. word_tokenize

    3. keep only tokens where token.isalpha() is True

    4. remove stop words using NLTK's English list minus {'not', 'no', 'nor'}

Report the final list as b2_clean_tokens .

In [11]:
nltk.download('stopwords')
from nltk.corpus import stopwords

para = data["basic"]["paragraph"]

# Step 1: Convert to lowercase
text = para.lower()

# Step 2: Tokenize
tokens = word_tokenize(text)

# Step 3: Keep only alphabetic tokens
tokens = [token for token in tokens if token.isalpha()]

# Step 4: Remove English stopwords, preserving not, no, and nor
stop_words = set(stopwords.words("english"))
stop_words -= {"not", "no", "nor"}

b2_clean_tokens = [
    token for token in tokens
    if token not in stop_words
]

print("b2_clean_tokens:", b2_clean_tokens)

b2_clean_tokens: ['mary', 'ca', 'read', 'message', 'says', 'words', 'broken', 'grace', 'says', 'engines', 'responding', 'fuel', 'running', 'low', 'nothing', 'normal', 'night', 'nobody', 'panicking']


[nltk_data] Downloading package stopwords to /home/cse/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# B3. Word decoder (stemming vs lemmatization)

data["basic"]["words"] holds [word, pos] pairs, where pos is the WordNet part-of-speech code to use ( n , v , a , r ).

For each word, find the Porter stem and the WordNet lemma (use the given pos ).

Report:

    b3_table : a dictionary like {"word": ["stem", "lemma"], ...}

    b3_overstemmed_pair : the two words in your list that the Porter stemmer wrongly merges into one stem, as a list of two words

In [12]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Initialize the stemmer and lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Store the results
b3_table = {}

for word, pos in data["basic"]["words"]:
    stem = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos)
    b3_table[word] = [stem, lemma]

print("b3_table:", b3_table)

b3_table: {'worse': ['wors', 'bad'], 'flies': ['fli', 'fly'], 'better': ['better', 'good'], 'caring': ['care', 'care'], 'university': ['univers', 'university'], 'universe': ['univers', 'universe']}


In [13]:
from collections import defaultdict

stem_groups = defaultdict(list)

for word, pos in data["basic"]["words"]:
    stem_groups[stemmer.stem(word)].append(word)

b3_overstemmed_pair = []

for stem, words in stem_groups.items():
    if len(words) > 1:
        b3_overstemmed_pair = words
        break

print("b3_overstemmed_pair:", b3_overstemmed_pair)

b3_overstemmed_pair: ['university', 'universe']


# B4. Signal strength (Bag of Words and TF-IDF)

data["basic"]["docs"] is a list of 4 tiny documents. Target word can be accessed
using data["basic"]["target"] is [word, doc_index] .

1. Run CountVectorizer() with default settings on the 4 documents. Report b4_vocab_size (number of unique words) and b4_row_sums (a list with the total word count of each document).

2. Calculate the TF-IDF of the target word using the formulas (TF = count / words in document, IDF = log₁₀(N / df)). Report b4_tfidf .

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

docs = data["basic"]["docs"]

# Create the Bag of Words representation
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)

# Number of unique words
b4_vocab_size = len(vectorizer.get_feature_names_out())

# Total word count in each document
b4_row_sums = X.sum(axis=1).A1.tolist()

print("b4_vocab_size:", b4_vocab_size)
print("b4_row_sums:", b4_row_sums)

b4_vocab_size: 7
b4_row_sums: [5, 5, 5, 5]


In [16]:
import math

target_word, doc_index = data["basic"]["target"]

# Select the target document
target_doc = docs[doc_index]

# Count the target word in the document
target_tokens = target_doc.lower().split()
word_count = target_tokens.count(target_word.lower())

# Calculate TF
tf = word_count / len(target_tokens)  # 1/5 = 0.2

# Calculate IDF
N = len(docs)
df = sum(
    1 for doc in docs
    if target_word.lower() in doc.lower().split()
)

idf = math.log10(N / df)  # log10(4/3)=0.125

# Calculate TF-IDF, rounded to 3 decimal places
b4_tfidf = round(tf * idf, 3)  # 0.2* 0.125 = 0.025

print("Target word:", target_word)
print("Document index:", doc_index)
print("TF:", tf)
print("DF:", df)
print("IDF:", idf)
print("b4_tfidf:", b4_tfidf)

Target word: oxygen
Document index: 1
TF: 0.2
DF: 3
IDF: 0.12493873660829992
b4_tfidf: 0.025


# B5. Crew manifest (encodings)
data["basic"]["roles"] is a list of various crew roles.

    Label-encode them with LabelEncoder and report b5_label_encoded (list of integers).
    
    One-hot encode them and report b5_onehot_shape as [rows, columns] .

In [17]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

roles = data["basic"]["roles"]

# Label encoding
label_encoder = LabelEncoder()
b5_label_encoded = label_encoder.fit_transform(roles).tolist()

# One-hot encoding
onehot_encoder = OneHotEncoder()
onehot_encoded = onehot_encoder.fit_transform(
    [[role] for role in roles]
)

# Get the shape: [rows, columns]
b5_onehot_shape = list(onehot_encoded.shape)

print("b5_label_encoded:", b5_label_encoded)
print("b5_onehot_shape:", b5_onehot_shape)

b5_label_encoded: [3, 2, 1, 2, 0, 1, 1, 3]
b5_onehot_shape: [8, 4]


# B6. Meaning radar (word vectors)

data["basic"]["vectors"] maps three words to 3-number embedding vectors.

Compute the cosine similarity for every pair of words (first with second, first with third, second with third, in the order they appear in the file). Use NumPy or plain Python, not a ready-made cosine function. Report:

    b6_cosines : a dictionary such as {"word1-word2": 0.123, ...}
    
    b6_most_similar : the key of the most similar pair

In [18]:
import math
from itertools import combinations

vectors = data["basic"]["vectors"]

def cosine_similarity(v1, v2):
    # Calculate the dot product
    dot_product = sum(a * b for a, b in zip(v1, v2))

    # Calculate the magnitudes
    magnitude1 = math.sqrt(sum(a * a for a in v1))
    magnitude2 = math.sqrt(sum(b * b for b in v2))

    # Calculate cosine similarity [ cosine_similarity(A,B)= A.B/|A||B| ]
    return dot_product / (magnitude1 * magnitude2)


# Compare pairs in the order they appear in the data
b6_cosines = {}

for word1, word2 in combinations(vectors.keys(), 2):
    similarity = cosine_similarity(vectors[word1], vectors[word2])
    b6_cosines[f"{word1}-{word2}"] = round(similarity, 3)

# Find the most similar pair
b6_most_similar = max(b6_cosines, key=b6_cosines.get)

print("b6_cosines:", b6_cosines)
print("b6_most_similar:", b6_most_similar)

b6_cosines: {'planet-crew': 0.895, 'planet-probe': 0.6, 'crew-probe': 0.43}
b6_most_similar: planet-crew


In [29]:
answers_basic = {
    "b1_sentences": b1_sentences,
    "b1_split_tokens": b1_split_tokens,
    "b1_nltk_tokens": b1_nltk_tokens,

    "b2_clean_tokens": b2_clean_tokens,

    "b3_table": b3_table,
    "b3_overstemmed_pair": b3_overstemmed_pair,

    "b4_vocab_size": b4_vocab_size,
    "b4_row_sums": b4_row_sums,
    "b4_tfidf": b4_tfidf,

    "b5_label_encoded": b5_label_encoded,
    "b5_onehot_shape": b5_onehot_shape,

    "b6_cosines": b6_cosines,
    "b6_most_similar": b6_most_similar
}

# MISSION LOG 2: Decoding the Galaxy (Advanced)

### Mary says: "Now the real transmissions. They are drowning in symbols, tags and links. Find the patterns, scrub the noise, and teach me what the crew is feeling."

#### Your data is log = data['advanced']['log'] : ten raw transmission lines.

# A1. Decode the stream (regex extraction)

Use re.findall to extract each of these from the whole log:

Item : stardate, signal, callsign, freq, email

For each item report a dictionary, for example a1_stardate = {"count": 10, "third":
"SD4550.63"} (the number of matches, and the third match in order of appearance). Do
the same for the keys a1_signal , a1_callsign , a1_freq and a1_email .

In [19]:
import re

log = data["advanced"]["log"]

# 1. Extract stardates
stardates = re.findall(r"SD\d{4}\.\d{2}", log)
a1_stardate = {
    "count": len(stardates),
    "third": stardates[2]
}

# 2. Extract signal IDs
signals = re.findall(r"SIG-[A-Z]{3}-\d{4}", log)
a1_signal = {
    "count": len(signals),
    "third": signals[2]
}

# 3. Extract callsigns, excluding email addresses
callsigns = re.findall(r"(?<![\w.+-])@[a-z_]+\d*", log)
a1_callsign = {
    "count": len(callsigns),
    "third": callsigns[2]
}

# 4. Extract frequencies
frequencies = re.findall(r"\d+\.\d{2} GHz", log)
a1_freq = {
    "count": len(frequencies),
    "third": frequencies[2]
}

# 5. Extract email addresses
emails = re.findall(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
    log
)
a1_email = {
    "count": len(emails),
    "third": emails[2]
}

print("a1_stardate:", a1_stardate)
print("a1_signal:", a1_signal)
print("a1_callsign:", a1_callsign)
print("a1_freq:", a1_freq)
print("a1_email:", a1_email)

a1_stardate: {'count': 10, 'third': 'SD4687.60'}
a1_signal: {'count': 10, 'third': 'SIG-VUN-8255'}
a1_callsign: {'count': 10, 'third': '@rocky89'}
a1_freq: {'count': 10, 'third': '16.58 GHz'}
a1_email: {'count': 10, 'third': 'ryland.g@hailmary.space'}


# A2. Scrub the noise (regex cleaning)

Write clean_log(text) that applies exactly these steps, in order:
    
    1. lowercase
    2. remove URLs ( https?://\S+ )
    3. remove HTML tags (lazy match)
    4. remove emails
    5. remove @callsigns
    6. delete every character that is not a letter or whitespace
    7. collapse repeated whitespace into one space and strip the ends

Run it on the whole log. Report a2_word_count (number of words) and a2_longest_word (if there is a tie, pick the alphabetically last word).

In [20]:
import re

def clean_log(text):
    # Step 1: Convert to lowercase
    text = text.lower()

    # Step 2: Remove URLs
    text = re.sub(r"https?://\S+", "", text)

    # Step 3: Remove HTML tags using a lazy match
    text = re.sub(r"<.*?>", "", text)

    # Step 4: Remove email addresses
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "",
        text
    )

    # Step 5: Remove @callsigns
    text = re.sub(r"@[a-z_]+\d*", "", text)

    # Step 6: Remove everything except letters and whitespace
    text = re.sub(r"[^a-z\s]", "", text)

    # Step 7: Collapse repeated whitespace and strip the ends
    text = re.sub(r"\s+", " ", text).strip()

    return text


log = data["advanced"]["log"]
quasar_clean = clean_log(log)

words = quasar_clean.split()

# Count words
a2_word_count = len(words)

# Longest word; alphabetical last wins ties
a2_longest_word = max(words, key=lambda word: (len(word), word))

print("a2_word_count:", a2_word_count)
print("a2_longest_word:", a2_longest_word)

a2_word_count: 195
a2_longest_word: understand


# A3. The greedy gremlin (greedy vs lazy)

Every message in the log sits inside <msg> ... </msg> tags, and some lines contain two messages.

Mary's old code was:
    
    re.findall(r"<msg>.*</msg>", log)

1. Run it and note how many matches it returns.
2. Write a corrected pattern that returns only the text inside each tag, one match per message.
3. Report a3_msg_count (number of messages) and a3_longest_msg (the longest message text; if tied, the alphabetically last).

In [21]:
import re

log = data["advanced"]["log"]

greedy_matches = re.findall(r"<msg>.*</msg>", log)

print("Greedy matches:", len(greedy_matches))

Greedy matches: 10


In [22]:
messages = re.findall(r"<msg>(.*?)</msg>", log, flags=re.DOTALL)

print("Corrected message count:", len(messages))

Corrected message count: 14


In [23]:

# Number of messages
a3_msg_count = len(messages)

# Longest message; alphabetical last wins ties
a3_longest_msg = max(messages, key=lambda msg: (len(msg), msg))

print("a3_msg_count:", a3_msg_count)
print("a3_longest_msg:", a3_longest_msg)

a3_msg_count: 14
a3_longest_msg: The antennas were catching faint radio waves from the stars.


# A4. Language module reboot (the NLTK pipeline)

Take the messages from A3 and apply this pipeline to each message, then pool all the resulting words together:

1. lowercase, then remove every character that is not a letter or whitespace ( re.sub )
2. word_tokenize
3. remove stop words (NLTK English list minus {'not', 'no', 'nor'} )
4. POS-tag with pos_tag , then lemmatize each word with WordNetLemmatizer using the tag converted as in notes section 1.4

Use FreqDist on the pooled lemmas. Report a4_top5 : the 5 most frequent lemmas as [lemma, count] pairs, sorted by count (highest first), then alphabetically when counts tie.

In [24]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import pos_tag, FreqDist
from nltk.stem import WordNetLemmatizer

# Extract individual messages from A3
messages = re.findall(r"<msg>(.*?)</msg>", log)

# Preserve the negation words
stop_words = set(stopwords.words("english")) - {"not", "no", "nor"}

lemmatizer = WordNetLemmatizer()

# Convert Penn Treebank tags to WordNet POS codes
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return "a"
    elif tag.startswith("V"):
        return "v"
    elif tag.startswith("N"):
        return "n"
    elif tag.startswith("R"):
        return "r"
    else:
        return "n"

# Process each message and pool the resulting lemmas
all_lemmas = []

for message in messages:
    # Step 1: Lowercase and remove nonletters
    cleaned = re.sub(r"[^a-z\s]", "", message.lower())

    # Step 2: Tokenize
    tokens = word_tokenize(cleaned)

    # Step 3: Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Step 4: POS-tag and lemmatize
    tagged_words = pos_tag(tokens)

    lemmas = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged_words
    ]

    all_lemmas.extend(lemmas)

# Calculate word frequencies
freq_dist = FreqDist(all_lemmas)

# Sort by descending frequency, then alphabetical order
sorted_lemmas = sorted(
    freq_dist.items(),
    key=lambda item: (-item[1], item[0])
)

# Keep the five most frequent lemmas
a4_top5 = [[word, count] for word, count in sorted_lemmas[:5]]

print("a4_top5:", a4_top5)

a4_top5: [['repair', 3], ['arrive', 2], ['computer', 2], ['corridor', 2], ['damage', 2]]


# A5. Friend or foe? (a mini classifier plus VADER)

data['advanced']['train'] is 8 labelled messages [text, label] (1 = friendly, 0 = hostile). data['advanced']['test'] is 3 unlabelled messages.

1. Preprocess every training and test message with the same pipeline as A4 (join the lemmas back into a string).
2. Fit TfidfVectorizer() on the training messages only, then transform the test messages.
3. Train MultinomialNB() and predict the 3 test messages. Report a5_model_pred (list of 0/1).
4. Run VADER on the raw, unprocessed test messages. Label each message 1 if compound >= 0.05 , 0 if compound <= -0.05 , and None otherwise. Report a5_vader .

In [25]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Get training and test data
train_data = data["advanced"]["train"]
test_data = data["advanced"]["test"]

train_texts = [text for text, label in train_data]
train_labels = [label for text, label in train_data]
test_texts = test_data

# Same preprocessing pipeline as A4
stop_words = set(stopwords.words("english")) - {"not", "no", "nor"}
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return "a"
    elif tag.startswith("V"):
        return "v"
    elif tag.startswith("N"):
        return "n"
    elif tag.startswith("R"):
        return "r"
    return "n"

def preprocess_message(message):
    # Lowercase and retain only letters and whitespace
    cleaned = re.sub(r"[^a-z\s]", "", message.lower())

    # Tokenize and remove stopwords
    tokens = word_tokenize(cleaned)
    tokens = [word for word in tokens if word not in stop_words]

    # POS-tag and lemmatize
    tagged_tokens = pos_tag(tokens)
    lemmas = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged_tokens
    ]

    # Join lemmas into a single string
    return " ".join(lemmas)


# 1. Preprocess training and test messages
train_processed = [preprocess_message(text) for text in train_texts]
test_processed = [preprocess_message(text) for text in test_texts]

# 2. Fit TF-IDF on training messages only
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_processed)
X_test = vectorizer.transform(test_processed)

# 3. Train Multinomial Naive Bayes and predict
model = MultinomialNB()
model.fit(X_train, train_labels)

a5_model_pred = model.predict(X_test).tolist()

print("a5_model_pred:", a5_model_pred)


# 4. Run VADER on the RAW test messages
sia = SentimentIntensityAnalyzer()

a5_vader = []

for message in test_texts:
    compound = sia.polarity_scores(message)["compound"]

    if compound >= 0.05:
        label = 1
    elif compound <= -0.05:
        label = 0
    else:
        label = None

    a5_vader.append(label)

print("a5_vader:", a5_vader)

a5_model_pred: [1, 0, 1]
a5_vader: [1, 0, 1]


In [31]:
answers_advanced = {
    # A1: Regex extraction
    "a1_stardate": a1_stardate,
    "a1_signal": a1_signal,
    "a1_callsign": a1_callsign,
    "a1_freq": a1_freq,
    "a1_email": a1_email,

    # A2: Regex cleaning
    "a2_word_count": a2_word_count,
    "a2_longest_word": a2_longest_word,

    # A3: Message extraction
    "a3_msg_count": a3_msg_count,
    "a3_longest_msg": a3_longest_msg,

    # A4: NLTK pipeline
    "a4_top5": a4_top5,

    # A5: Classification and VADER
    "a5_model_pred": a5_model_pred,
    "a5_vader": a5_vader
}

Captain's Debrief (both logs)

Answer in markdown cells at the end of your notebook:
1. Prediction log. Before running B3 (or A2, for the advanced log), write down your guess for one or two outputs. After running, say what matched and what surprised you.
2. Biggest bug. Paste the exact error message or wrong output you got at some point, and what fixed it. If you had no errors, say so honestly.
3. Help declaration. List any help you used (documentation, classmates, AI tools) and for what. Write "none" if none.

# Answer

## 1. Prediction Log

### B3 - Stemming and Lemmatization

Before running B3, I predicted that stemming and lemmatization would produce
similar results for some words but differ for others. I expected lemmatization
to produce more meaningful dictionary words.

After running the code, I observed that the Porter stemmer reduced both
"university" and "universe" to "univers". This surprised me because the
words have different meanings. It showed me that stemming can remove word
endings without considering the word's actual meaning.

### A2 - Advanced Log Preprocessing

Before running A2, I predicted that removing metadata, URLs, email addresses,
and punctuation would make the transmission log easier to analyze.

After running the code, I observed that the cleaned text contained the message
content without the unwanted metadata. I also learned that preprocessing
choices, such as which characters to remove, can affect the resulting word
count.

## 2. Biggest Bug

One unexpected output occurred during B2 preprocessing. The word "can't"
was split into "ca" and "n't" during tokenization. After removing stopwords
and filtering tokens, "ca" remained in the output.

This happened because contractions can be split into multiple tokens,
and removing stopwords does not necessarily remove every fragment of a
contraction. Examining the tokenization output helped me understand how
the order and choice of preprocessing steps affect the final result.

## 3. Help Declaration

ChatGPT (AI tool): Used for understanding NLP concepts, including
  tokenization, stemming, lemmatization, TF-IDF, label encoding, cosine
  similarity, regular expressions, text preprocessing, Naive Bayes
  classification, and VADER sentiment analysis. Also used for code guidance,
  interpreting outputs, and understanding unexpected preprocessing behavior.
  Used to inspect outputs, and verify the results.

I am responsible for understanding the methods, checking the outputs, and
ensuring that my submitted work follows the assignment's rules.

Submission checklist
1. Notebook link shared (view access), outputs visible, version history intact
2. Boot-up cell output (NLTK version) visible
3. answers_basic and/or answers_advanced printed and pasted into the form
4. Captain's Debrief completed, including the help declaration
Mary says: "Thank you, crew member. Every word you teach me brings us closer to
home."

In [32]:
print(answers_basic)
print(answers_advanced)

{'b1_sentences': ["Mary can't read the message; she says the words are broken.", "Dr. Grace says the engines aren't responding, and fuel is running low!", 'Nothing was normal about that night, but nobody was panicking.'], 'b1_split_tokens': ['Mary', "can't", 'read', 'the', 'message;', 'she', 'says', 'the', 'words', 'are', 'broken.', 'Dr.', 'Grace', 'says', 'the', 'engines', "aren't", 'responding,', 'and', 'fuel', 'is', 'running', 'low!', 'Nothing', 'was', 'normal', 'about', 'that', 'night,', 'but', 'nobody', 'was', 'panicking.'], 'b1_nltk_tokens': ['Mary', 'ca', "n't", 'read', 'the', 'message', ';', 'she', 'says', 'the', 'words', 'are', 'broken', '.', 'Dr.', 'Grace', 'says', 'the', 'engines', 'are', "n't", 'responding', ',', 'and', 'fuel', 'is', 'running', 'low', '!', 'Nothing', 'was', 'normal', 'about', 'that', 'night', ',', 'but', 'nobody', 'was', 'panicking', '.'], 'b2_clean_tokens': ['mary', 'ca', 'read', 'message', 'says', 'words', 'broken', 'grace', 'says', 'engines', 'responding